# Notebook 01: Eigenstrain Extraction from DFT Structures

Extracts eigenstrain tensors for point defect configurations in Fe-doped SrTiO₃ (STO)
by comparing DFT-relaxed defective structures to a pristine reference lattice.

**Method:** Displacement vectors $u_i(\mathbf{r})$ are computed by matching atoms
between reference and defective structures. A linear regression on the displacement
field within a local region around each defect yields the displacement gradient tensor
$\partial u_i / \partial x_j$, whose symmetric part is the eigenstrain:

$$\varepsilon^*_{ij} = \frac{1}{2}\left(\frac{\partial u_i}{\partial x_j} + \frac{\partial u_j}{\partial x_i}\right)$$

**Output:** A DataFrame of eigenstrain components for each defect configuration,
used as input to all subsequent notebooks.

In [4]:
import os
import numpy as np
import pandas as pd
from pymatgen.io.vasp import Poscar
from scipy.spatial import cKDTree
from sklearn.linear_model import LinearRegression

# --- Paths to DFT structure files (not included in repo; available on request) ---
# Update to your local copies:
REFERENCE_PATH = "/path/to/STO_pristine.vasp"
DEFECT_BASE_PATH = "/path/to/defect_calculations/"

In [5]:
def load_structure(file_path):
    """Load a VASP POSCAR/CONTCAR file via pymatgen."""
    return Poscar.from_file(file_path).structure


def match_atoms_by_position(ref_structure, def_structure, tolerance=0.5):
    """
    Match atoms between reference and defective structures by nearest-neighbor
    search. Returns matched coordinates and displacement vectors.
    """
    ref_coords = ref_structure.cart_coords
    def_coords = def_structure.cart_coords
    tree = cKDTree(def_coords)
    distances, indices = tree.query(ref_coords, distance_upper_bound=tolerance)
    matched = distances < tolerance
    displacements = def_coords[indices[matched]] - ref_coords[matched]
    return ref_coords[matched], def_coords[indices[matched]], displacements


def fit_strain_tensor(ref_pos, disp_vec):
    """
    Fit displacement gradient tensor via linear regression on local
    displacement field. Returns symmetric strain tensor and gradient.
    """
    model = LinearRegression().fit(ref_pos, disp_vec)
    grad_u = model.coef_
    strain_tensor = 0.5 * (grad_u + grad_u.T)
    return strain_tensor, grad_u


def extract_all_eigenstrains(reference_path, defect_base_path, tolerance=0.5):
    """
    Iterate over all defect subdirectories, extract eigenstrain tensor for each,
    and return a summary DataFrame.
    """
    ref_structure = load_structure(reference_path)
    results = []

    for dirname in sorted(os.listdir(defect_base_path)):
        defect_dir = os.path.join(defect_base_path, dirname)
        poscar_path = os.path.join(defect_dir, "CONTCAR")
        if not os.path.isdir(defect_dir) or not os.path.exists(poscar_path):
            continue
        try:
            def_structure = load_structure(poscar_path)
            ref_pos, _, disp_vec = match_atoms_by_position(ref_structure, def_structure, tolerance)
            strain_tensor, _ = fit_strain_tensor(ref_pos, disp_vec)
            results.append({
                "defect": dirname,
                "strain_xx": strain_tensor[0, 0],
                "strain_yy": strain_tensor[1, 1],
                "strain_zz": strain_tensor[2, 2],
                "strain_xy": strain_tensor[0, 1],
                "strain_yz": strain_tensor[1, 2],
                "strain_xz": strain_tensor[0, 2],
                "norm_disp": np.mean(np.linalg.norm(disp_vec, axis=1))
            })
        except Exception as e:
            print(f"Skipped {dirname}: {e}")

    return pd.DataFrame(results)

In [6]:
df_eigenstrain = extract_all_eigenstrains(REFERENCE_PATH, DEFECT_BASE_PATH)
df_eigenstrain.to_csv("../data/eigenstrain_results.csv", index=False)
print(f"Extracted eigenstrains for {len(df_eigenstrain)} defect configurations.")
df_eigenstrain.head(10)

Extracted eigenstrains for 16 defect configurations.


,defect,strain_xx,strain_yy,strain_zz,strain_xy,strain_yz,strain_xz,norm_disp
0,2Fe_Ti_+1_3_+U,0.008202,0.008202,0.005796,-3.037690e-05,-0.000014,-0.000014,0.068403
1,2Fe_Ti_+2_3_+U,0.008305,0.008305,0.005874,-3.177804e-05,-0.000015,-0.000015,0.068567
2,2Fe_Ti_-1_3_+U,0.005683,0.005683,0.005660,1.243954e-04,0.000057,0.000057,0.064885
3,2Fe_Ti_-2_3_+U,0.006383,0.006383,0.006353,-5.684799e-06,-0.000005,-0.000005,0.064995
4,2Fe_Ti_0_3_+U,0.007076,0.007076,0.005981,-1.509187e-05,-0.000008,-0.000008,0.063666
5,2Fe_Ti_V_O_+1_3_+U,0.006421,0.006421,0.004700,4.880228e-07,0.000002,0.000002,0.076459
6,2Fe_Ti_V_O_+2_3_+U,0.005922,0.005922,0.004977,7.220869e-04,0.000354,0.000354,0.070726
7,2Fe_Ti_V_O_0_3_+U,0.005691,0.005691,0.004834,-2.921913e-07,0.000004,0.000004,0.078955
8,Fe_Ti_-1_3_+U,0.005460,0.005460,0.005460,4.115909e-05,0.000041,0.000041,0.063102
9,Fe_Ti_0_3_+U,0.006070,0.006070,0.006070,7.751185e-05,0.000078,0.000078,0.063642


In [7]:
# Defect configurations used in subsequent notebooks
key_defects = [
    "Fe_Ti_V_O_0_3_+U",
    "2Fe_Ti_V_O_0_3_+U",
    "Fe_Ti_0_3_+U",
    "VO_0_3_+U",
    "2Fe_Ti_0_3_+U"
]

df_eigenstrain[df_eigenstrain["defect"].isin(key_defects)][
    ["defect", "strain_xx", "strain_yy", "strain_xy"]
].reset_index(drop=True)

,defect,strain_xx,strain_yy,strain_xy
0,2Fe_Ti_0_3_+U,0.007076,0.007076,-1.509187e-05
1,2Fe_Ti_V_O_0_3_+U,0.005691,0.005691,-2.921913e-07
2,Fe_Ti_0_3_+U,0.006070,0.006070,7.751185e-05
3,Fe_Ti_V_O_0_3_+U,0.006373,0.006373,7.435702e-04
4,VO_0_3_+U,0.005488,0.003791,1.314178e-04
